# Step 6: Model building and selection

One notebook, top to bottom:

| Step | What | Technique (package) |
|---|---|---|
| 0 | Load `model_table.parquet`, build predictors, split train / validation / test | DuckDB, pandas |
| 1 | Check the OLS assumptions on the all-candidate model | Residual plots, normal QQ plots (`sm.qqplot`), Breusch-Pagan (`het_breuschpagan`) |
| 2 | Remove collinear predictors | Auxiliary regressions with overall and partial F-tests (`anova_lm`), joint vs individual F-tests (`f_test`). **No VIF** |
| 3 | Keep only significant predictors | Day-clustered F-tests per term (`wald_test_terms`), backward elimination |
| 4 | Fit the full model (significant, non-collinear predictors) | `smf.ols` with day-clustered standard errors |
| 5 | Create new models with stepwise selection | Forward and backward stepwise on `results.bic` |
| 6 | Evaluate every model | `rsquared_adj`, `aic`, `bic` (training); `rmse`, `meanabs` (validation) |
| 7 | Final check of the chosen model on the untouched test weeks | RMSE / MAE vs the last-week baseline, residual + QQ plots |

**Inputs.** Run the pipeline first (`scripts/01_` to `04_`, see `CLAUDE.md`). This notebook reads only
`data/processed/model_table.parquet`, through DuckDB (no `pyarrow` needed). Paths are relative to `scripts/`.
A full run takes several minutes; most of it is the stepwise search in Step 5.

**Design decisions** (same as `05_statistical_tests.ipynb`):
- **Zones:** the 50 busiest zones, ranked by **training** pickups.
- **Target:** `log(pickups + 1)`. Step 1 shows why.
- **Split:** within every month, the last 7 days are **test**, the 7 days before are **validation**, the rest is
  **train**. Every month has training rows, so month dummies always have a coefficient.
- **Standard errors:** weather and holidays are the same for every zone in a given hour, so rows from the same day
  are not independent. All significance tests use standard errors **clustered by day**.

## Step 0: Settings, data and predictors

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pandas.tseries.holiday import USFederalHolidayCalendar
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.tools.eval_measures import meanabs, rmse

MODEL_TABLE = Path("../data/processed/model_table.parquet")
TOP_ZONES = 50        # busiest zones by training pickups (the top 60 carry ~90% of all pickups)
ALPHA = 0.05          # significance level for every test
AUX_R2_FLAG = 0.8     # auxiliary R² above this = serious collinearity
PAIR_CORR_FLAG = 0.7  # |r| above this = test the pair jointly vs individually
SEED = 860

INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "text.color": INK, "xtick.color": INK_2, "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "figure.dpi": 110,
})
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)

**Load.** One DuckDB query: assign the split, pick the top zones on training pickups, derive the calendar columns,
and add two lagged demand values for the same zone: **same clock hour yesterday** (`lag24`) and **same clock hour
last week** (`lag168`). Both are known a day ahead, so they can be used for next-day dispatch planning without
leaking the answer. They are the "historic pickup activity" from the proposal.

Rows are dropped when a lag or the weather is missing: the first week of January (no lag one week back), hours whose
lag falls on the skipped 2025-03-09 02:00, and 2025-12-31 23:00 (no shifted weather).

In [ ]:
con = duckdb.connect()
df = con.execute(f"""
    WITH m AS (SELECT * FROM '{MODEL_TABLE.as_posix()}'),
    s AS (
        SELECT *, CASE WHEN day(hour) > day(last_day(hour)) - 7  THEN 'test'
                       WHEN day(hour) > day(last_day(hour)) - 14 THEN 'val'
                       ELSE 'train' END AS split
        FROM m
    ),
    top AS (
        SELECT zone FROM s WHERE split = 'train'
        GROUP BY zone ORDER BY sum(pickups) DESC LIMIT {TOP_ZONES}
    )
    SELECT s.zone, s.hour, s.split, s.pickups,
           hour(s.hour) AS hour_of_day, isodow(s.hour) AS dow, month(s.hour) AS month,
           s.hour::DATE AS date, dayofyear(s.hour) AS day,
           s.temperature_2m, s.precipitation, s.snowfall, s.snow_depth, s.cloud_cover, s.shortwave_radiation,
           s.is_day, l24.pickups AS lag24, l168.pickups AS lag168
    FROM s
    JOIN top USING (zone)
    LEFT JOIN m l24  ON l24.zone  = s.zone AND l24.hour  = s.hour - INTERVAL 24 HOUR
    LEFT JOIN m l168 ON l168.zone = s.zone AND l168.hour = s.hour - INTERVAL 168 HOUR
    ORDER BY s.zone, s.hour
""").df()

n_raw = len(df)
df = df.dropna().reset_index(drop=True)
df = df.astype({c: "float64" for c in df.columns if str(df[c].dtype) in ("Float64", "Int64")})  # nullable -> numpy
df["date"] = pd.to_datetime(df["date"])
print(f"{n_raw:,} zone-hours for the top {TOP_ZONES} zones; dropped {n_raw - len(df):,} with a missing lag or weather")
print(df["split"].value_counts().reindex(["train", "val", "test"]).to_string())

**Predictors.** Each candidate is a *term* in a `statsmodels` formula: a single column, or a block of dummies (`C()`)
that is added, tested and dropped as a whole. The target is written into the formula as `np.log1p(pickups)`.

| Term | Formula | Notes |
|---|---|---|
| `hour`, `dow`, `month`, `zone` | `C(...)` | reference levels: hour 0, Monday, January, the busiest zone |
| `holiday` | `holiday` | US federal holiday (pandas `USFederalHolidayCalendar`) |
| `hour_x_weekend` | `C(hour_wknd)` | separate daily curve on weekends: `hour_wknd` = the hour on weekends, 0 otherwise. Weekend hour 0 shares the reference level, because 24 weekend-hour dummies would add up to `dow_6 + dow_7` (perfect collinearity) |
| `zone_x_weekend` | `C(zone_wknd)` | each zone's own weekend shift, coded the same way (the reference zone shares the reference level) |
| `temperature_2m`, `temp_sq` | °C, (°C − training mean)² | the square is centred so it isn't collinear with temperature itself; tests whether demand rises in both cold and heat |
| `precipitation`, `snowfall`, `snow_depth`, `cloud_cover`, `shortwave_radiation`, `is_day` | weather | |
| `log_lag24`, `log_lag168` | log(1 + lagged pickups) | logged to match the target |

Interactions follow the **hierarchy rule**: an interaction is only in a model if its main effects are
(`hour_x_weekend` needs `hour` and `dow`, `zone_x_weekend` needs `zone` and `dow`, `temp_sq` needs `temperature_2m`).

In [ ]:
holidays = USFederalHolidayCalendar().holidays("2025-01-01", "2025-12-31")
REF_ZONE = int(df[df["split"] == "train"].groupby("zone")["pickups"].sum().idxmax())
weekend = df["dow"] >= 6
df["holiday"] = df["date"].isin(holidays).astype(float)
df["hour_wknd"] = np.where(weekend, df["hour_of_day"], 0)
df["zone_wknd"] = np.where(weekend & (df["zone"] != REF_ZONE), df["zone"], 0)
df["log_lag24"] = np.log1p(df["lag24"])
df["log_lag168"] = np.log1p(df["lag168"])
df["temp_sq"] = (df["temperature_2m"] - df.loc[df["split"] == "train", "temperature_2m"].mean()) ** 2

train, val, test = (df[df["split"] == s].reset_index(drop=True) for s in ["train", "val", "test"])

LOG_TARGET = "np.log1p(pickups)"
CALENDAR = ["hour", "dow", "month", "holiday"]
CONTINUOUS = ["temperature_2m", "temp_sq", "precipitation", "snowfall", "snow_depth", "cloud_cover",
              "shortwave_radiation", "is_day", "log_lag24", "log_lag168"]
INTERACTIONS = ["hour_x_weekend", "zone_x_weekend"]
ALL_TERMS = CALENDAR + ["zone"] + CONTINUOUS + INTERACTIONS
TERMS = {"hour": "C(hour_of_day)", "dow": "C(dow)", "month": "C(month)",
         "zone": f"C(zone, Treatment({REF_ZONE}))", "hour_x_weekend": "C(hour_wknd)", "zone_x_weekend": "C(zone_wknd)",
         **{t: t for t in ["holiday"] + CONTINUOUS}}
TERM_NAME = {f: t for t, f in TERMS.items()}  # formula term -> short name
PARENTS = {"hour_x_weekend": ["hour", "dow"], "zone_x_weekend": ["zone", "dow"], "temp_sq": ["temperature_2m"]}

def formula(terms, target=LOG_TARGET):
    return f"{target} ~ " + (" + ".join(TERMS[t] for t in terms) if terms else "1")

def ols(terms, data=train, target=LOG_TARGET, clustered=False):
    """OLS from a formula. clustered=True: standard errors clustered by day, F/t reference distributions."""
    model = smf.ols(formula(terms, target), data)
    if clustered:
        return model.fit(cov_type="cluster", cov_kwds={"groups": data["day"]}, use_t=True)
    return model.fit()

def term_tests(res):
    """Joint F-test of each term's coefficients (dummy blocks together), using the fit's covariance."""
    t = res.wald_test_terms(skip_single=False, scalar=True).summary_frame()
    t = t.drop(index="Intercept").rename(index=TERM_NAME)
    return t.rename(columns={"P>F": "p", "df constraint": "df"})[["df", "F", "p"]]

def with_hierarchy(terms):
    """Drop any interaction whose main effects are no longer in the model."""
    terms = list(terms)
    while any(t in PARENTS and not set(PARENTS[t]) <= set(terms) for t in terms):
        terms = [t for t in terms if not (t in PARENTS and not set(PARENTS[t]) <= set(terms))]
    return terms

print(f"{len(holidays)} federal holidays; reference zone {REF_ZONE}")
print(f"train {len(train):,}, val {len(val):,}, test {len(test):,} rows")
print(formula(ALL_TERMS))

## Step 1: Check the assumptions with residual plots and normal QQ plots

OLS inference assumes the errors have **constant variance** (homoskedasticity) and are roughly **normal**. Fit the
model with every candidate term twice, on raw `pickups` and on `log(pickups + 1)`, and look at:

- **Residuals vs fitted** (random 20k points). A funnel shape = the spread changes with demand.
- **Residual SD by fitted-value decile.** A flat line = constant variance.
- **Normal QQ plot** of standardised residuals (random 5k points). Points on the dashed line = normal errors.
- **Breusch-Pagan test.** H₀ = constant variance. With 228k rows it rejects for tiny departures, so judge the size
  of the problem from the plots.

In [ ]:
def diagnostics(axes, fitted, resid, label):
    fitted, resid = pd.Series(np.asarray(fitted)), pd.Series(np.asarray(resid))
    s = fitted.sample(20_000, random_state=SEED).index
    ax = axes[0]
    ax.scatter(fitted[s], resid[s], s=4, alpha=0.25, color=BLUE, linewidths=0)
    ax.axhline(0, color=INK_2, lw=1)
    ax.set(title=f"{label}: residuals vs fitted", xlabel="fitted", ylabel="residual")

    decile = pd.qcut(fitted, 10, labels=False, duplicates="drop")
    by_decile = pd.DataFrame({"fitted": fitted, "resid": resid}).groupby(decile).agg(
        mean_fitted=("fitted", "mean"), resid_sd=("resid", "std"))
    ax = axes[1]
    ax.plot(by_decile["mean_fitted"], by_decile["resid_sd"], marker="o", ms=5, color=BLUE)
    ax.set(title=f"{label}: residual SD by fitted decile", xlabel="mean fitted value in decile", ylabel="residual SD")
    ax.set_ylim(bottom=0)

    ax = axes[2]
    sm.qqplot(resid.sample(5_000, random_state=SEED), fit=True, line="45", ax=ax,
              marker=".", markersize=3, alpha=0.5, markerfacecolor=BLUE, markeredgecolor=BLUE)
    ax.get_lines()[1].set(color=INK_2, linewidth=1.2, linestyle="--")
    ax.set(title=f"{label}: normal QQ plot", xlabel="theoretical quantile", ylabel="standardised residual")

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
rows = []
for row, (target, label) in enumerate([("pickups", "Raw pickups"), (LOG_TARGET, "log(pickups + 1)")]):
    res = ols(ALL_TERMS, target=target)
    diagnostics(axes[row], res.fittedvalues, res.resid, label)
    bp_lm, bp_p, _, _ = het_breuschpagan(res.resid, res.model.exog)
    rows.append({"target": label, "R2": res.rsquared, "BP LM": bp_lm, "BP p": bp_p, "skew": res.resid.skew(),
                 "excess kurtosis": res.resid.kurt(), "negative fitted": (res.fittedvalues < 0).mean()})
    del res
fig.tight_layout()
plt.show()
pd.DataFrame(rows).set_index("target")

**Results.**
- **Raw pickups break the assumptions badly.** The residuals fan out: their SD rises from about 14 to about 97 pickups
  from the lowest to the highest decile. 13% of fitted values are **negative pickups**. The QQ plot has a long right
  tail (skew 1.4, excess kurtosis 7.6).
- **`log(pickups + 1)` is much better but not perfect.** Only 0.02% of fitted values are below zero, and the
  residuals are closer to symmetric (skew −0.4). The variance now runs the other way: the SD falls from about 0.63 in
  the quietest decile to about 0.22 in the busiest, because quiet zone-hours are small counts (the diagonal stripes
  are 0, 1, 2, ... pickups). Both QQ tails are heavier than normal, mostly the left tail (hours far below forecast).
- **Breusch-Pagan rejects constant variance for both** (p ≈ 0).

**Decision:** use `log(pickups + 1)` as the target. The heteroskedasticity that is left is handled with **day-clustered
standard errors** in every test below; they are valid under unequal variance and within-day correlation. The tails
are not normal, but with 228k rows the coefficient estimates are close to normal anyway (central limit theorem), so the
F-tests remain usable.

## Step 2: Remove collinear predictors with F-tests (no VIF)

Collinearity means a predictor carries almost the same information as other predictors. Its coefficient gets a large
standard error and can flip sign. Three checks, all on the training rows:

1. **Correlation matrix** of the continuous predictors (for reference).
2. **Auxiliary regressions (Farrar-Glauber F-test).** Regress each continuous predictor on all other main-effect
   predictors (calendar dummies, zone dummies, other continuous predictors). Report the auxiliary R² and its overall F,
   plus a **partial F** from `anova_lm` (calendar + zone only vs calendar + zone + other continuous predictors): does
   the rest of the weather and lag information still predict it once the calendar and zone are known? With 228k rows
   every F rejects, so the **size** of R² decides: above 0.8 is serious, and the predictor is **flagged**.
3. **Joint vs individual F-tests** on the all-candidate log model, with day-clustered standard errors. For every pair
   with |r| > 0.7: if the joint test that both coefficients are 0 rejects but an individual test does not, the pair
   shares one piece of information and the weaker one is dropped. A **flagged** predictor is dropped if its own
   F-test (given every other predictor) is not significant: its information is already in the model.

In [ ]:
corr = train[CONTINUOUS].corr()
corr.round(2)

In [ ]:
base = CALENDAR + ["zone"]
rows = []
for v in CONTINUOUS:
    others = [c for c in CONTINUOUS if c != v]
    restricted = ols(base, target=v)
    full = ols(base + others, target=v)
    partial = anova_lm(restricted, full).iloc[1]
    rows.append({"predictor": v, "R2 calendar+zone": restricted.rsquared, "aux R2 (all)": full.rsquared,
                 "overall F": full.fvalue, "overall p": full.f_pvalue,
                 "partial R2 (others)": 1 - full.ssr / restricted.ssr,
                 "partial F": partial["F"], "partial p": partial["Pr(>F)"], "flagged": full.rsquared > AUX_R2_FLAG})
del restricted, full
aux = pd.DataFrame(rows).set_index("predictor")
aux

In [ ]:
res_all = ols(ALL_TERMS, clustered=True)
single = term_tests(res_all)

drop, decisions = set(), []
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
for (a, b), r in pairs[pairs.abs() > PAIR_CORR_FLAG].items():
    joint = float(res_all.f_test(f"{a} = 0, {b} = 0").pvalue)
    pa, pb = single.loc[a, "p"], single.loc[b, "p"]
    collinear = joint < ALPHA and max(pa, pb) >= ALPHA
    weaker = a if pa >= pb else b
    if collinear:
        drop.add(weaker)
    decisions.append({"check": f"pair {a} / {b} (r = {r:.2f})", "joint p": joint,
                      "individual p": f"{pa:.4f} / {pb:.4f}",
                      "decision": f"drop {weaker}" if collinear else "keep both (each significant on its own)"})
for v in aux.index[aux["flagged"]]:
    p = single.loc[v, "p"]
    if p >= ALPHA:
        drop.add(v)
    decisions.append({"check": f"flagged {v} (aux R² = {aux.loc[v, 'aux R2 (all)']:.2f})",
                      "individual p": f"{p:.4f}",
                      "decision": f"drop {v}" if p >= ALPHA else "keep (adds significant information)"})
del res_all

STEP2_TERMS = with_hierarchy([t for t in ALL_TERMS if t not in drop])
print("dropped for collinearity:", sorted(set(ALL_TERMS) - set(STEP2_TERMS)) or "none")
pd.DataFrame(decisions).set_index("check")

**Results.**
- **Correlations:** only the two lags are strongly correlated (r = 0.88). `shortwave_radiation` and `is_day` come
  next (0.68).
- **Auxiliary regressions:** five predictors are flagged (aux R² > 0.8). The partial R² shows where the overlap comes
  from:
  - `temperature_2m` (0.90) and `is_day` (0.82) are almost entirely explained by **calendar dummies** (season and
    time of day). The other predictors add only 0.09 and 0.006.
  - `log_lag24` and `log_lag168` (0.83, 0.82) are explained largely **by each other** (partial R² 0.36).
  - `shortwave_radiation` (0.83) overlaps with both the calendar and `is_day`.
- **Decisions:**
  - The two lags are each significant on their own, so both are kept despite r = 0.88.
  - `temperature_2m` (p = 0.07) and `is_day` (p = 0.15) add no significant information beyond the calendar and the
    lags, so they are **dropped**. `temp_sq` goes with `temperature_2m` (hierarchy rule).
  - `shortwave_radiation` (p = 0.02) is kept for now.

Note: dropping temperature here only means it **doesn't help predict** zone-hour demand once the calendar and last
week's demand are known. Its effect on demand is still estimated in the FWL section of `05_statistical_tests.ipynb`.

## Step 3: Keep only significant predictors

Start from the terms that survived Step 2 and run **backward elimination**: test every term with a day-clustered
F-test (`wald_test_terms`, dummy blocks tested jointly), drop the least significant term if its p-value is above
0.05, refit, and repeat until every term is significant. Main effects are not dropped while an interaction that needs
them is still in.

In [ ]:
terms, log = list(STEP2_TERMS), []
while True:
    tests = term_tests(ols(terms, clustered=True))
    protected = {p for t in terms if t in PARENTS for p in PARENTS[t]}
    candidates = tests.drop(index=[t for t in protected if t in tests.index])
    worst = candidates["p"].idxmax()
    if candidates.loc[worst, "p"] <= ALPHA:
        break
    log.append({"dropped": worst, "p": candidates.loc[worst, "p"]})
    terms = with_hierarchy([t for t in terms if t != worst])

FULL_TERMS = terms
print("dropped as not significant, in order:")
print(pd.DataFrame(log).to_string(index=False) if log else "none")
tests.sort_values("p")

**Results.** Five terms are dropped, one at a time: `cloud_cover` (p = 0.28), `shortwave_radiation` (0.23),
`snowfall` (0.17), `snow_depth` (0.12) and `holiday` (0.07). Every remaining term is significant at the 1% level.

- **Weather:** after the calendar and last week's demand, only `precipitation` is significant.
- **Snow:** 2025 had few snowy hours in the training days, so the snow effects are measured imprecisely.
- **`holiday`:** borderline. Only 5 holidays fall in the training days, which gives few independent days to estimate
  it from.

## Step 4: The full model

The full model contains every term that passed Steps 2 and 3. Coefficients with day-clustered standard errors are shown
for the single-column terms. In a log model, `100 × (exp(b) − 1)` is the % change in pickups for a one-unit increase.
For the lags it is an elasticity: a 1% rise in last week's pickups goes with a `b`% rise now.

In [ ]:
res_full = ols(FULL_TERMS, clustered=True)
print(f"full model: {len(FULL_TERMS)} terms, {len(res_full.params)} parameters, "
      f"R² = {res_full.rsquared:.4f}, adjusted R² = {res_full.rsquared_adj:.4f}")
print(formula(FULL_TERMS))

single_cols = [t for t in FULL_TERMS if TERMS[t] == t]
coef = pd.DataFrame({
    "coef": res_full.params[single_cols], "clustered SE": res_full.bse[single_cols],
    "t": res_full.tvalues[single_cols], "p": res_full.pvalues[single_cols]})
coef["% change per unit"] = 100 * np.expm1(coef["coef"])
coef.loc[coef.index.str.startswith("log_lag"), "% change per unit"] = np.nan  # lags: read coef as an elasticity
coef

**Results.** The full model has 9 terms and 165 parameters, with R² = 0.920 (adjusted R² 0.920).
- **Rain:** each mm of precipitation in the hour raises pickups by about **2.3%** (clustered p = 0.002), after the
  calendar, zone and last week's demand are accounted for.
- **Last week's demand matters more than yesterday's:** the lag elasticities are 0.57 (same hour last week) vs 0.25
  (same hour yesterday). Last week matches the zone, weekday and hour exactly.

## Step 5: Stepwise selection creates new models

**Bidirectional stepwise selection** over the full model's terms, scored by **BIC** (it penalises extra
parameters more than AIC: ln(228k) ≈ 12.3 per parameter vs 2). At every step, try adding each term that is out and
removing each term that is in, fit each candidate with `smf.ols`, and make the move that lowers `results.bic` the
most. Stop when no move lowers BIC. The hierarchy rule still holds. (`statsmodels` has no stepwise routine, so the
search loop is written here; every fit and score comes from `statsmodels`.)

- **Forward** starts from the intercept only. Every step is saved as a new model (S1, S2, ...), giving a ladder of
  nested models from the single most useful term up.
- **Backward** starts from the full model and removes terms while that lowers BIC.

In [ ]:
bic_cache = {}
def bic(terms):
    key = frozenset(terms)
    if key not in bic_cache:
        bic_cache[key] = ols(terms).bic
    return bic_cache[key]

def stepwise(pool, start):
    current, path = list(start), []
    best = bic(current)
    while True:
        moves = [current + [t] for t in pool if t not in current] + [[c for c in current if c != t] for t in current]
        moves = [m for m in moves if with_hierarchy(m) == m]
        if not moves:
            break
        nxt = min(moves, key=bic)
        if bic(nxt) >= best:
            break
        added, removed = set(nxt) - set(current), set(current) - set(nxt)
        change = f"+ {added.pop()}" if added else f"- {removed.pop()}"
        current, best = nxt, bic(nxt)
        path.append({"change": change, "terms": list(current), "BIC": best})
    return path

forward = stepwise(FULL_TERMS, [])
backward = stepwise(FULL_TERMS, FULL_TERMS)
for name, path in [("forward", forward), ("backward", backward)]:
    print(f"{name}: {len(path)} steps")
    for i, p in enumerate(path, 1):
        print(f"  {i:>2}. {p['change']:<22} BIC = {p['BIC']:,.0f}")
print(f"{len(bic_cache)} models fitted")
print("forward stepwise ends at the full model" if set(forward[-1]["terms"]) == set(FULL_TERMS)
      else "forward stepwise ends at a different model")

**Results.**
- **Forward stepwise:** adds the terms in this order: last week's demand, yesterday's demand, month, hour, zone, day
  of week, the weekend daily curve, zone × weekend, precipitation. It ends exactly at the full model.
- **Backward stepwise:** removes nothing, so the full model is BIC-optimal among these terms. The Step 3 significance
  tests and BIC agree.
- **New models:** the forward path gives nine nested models, **S1 to S9** (S9 = the full model), compared in Step 6.

## Step 6: Evaluate the models

Every distinct model from Steps 4 and 5 is scored on:
- **Adjusted R², AIC, BIC** on the training rows (log scale). Higher adjusted R², lower AIC / BIC = better fit after
  the penalty for parameters.
- **RMSE and MAE on the validation weeks**, in pickups per zone-hour (predictions converted back with
  `exp(ŷ) − 1`). This is the out-of-sample check that picks the model. `RMSE_val_log` is the same on the log scale.

The **last-week baseline** (predict the pickups from the same zone and clock hour 7 days earlier) is the bar a
model has to clear.

In [ ]:
models = {f"S{i}: {p['change']}": p["terms"] for i, p in enumerate(forward, 1)}
models["Full model"] = FULL_TERMS
if backward:
    models["Backward stepwise"] = backward[-1]["terms"]

names = {}  # one row per distinct model; duplicate names are joined with " = "
for name, terms in models.items():
    names.setdefault(frozenset(terms), []).append(name)

rows = []
for key, labels in names.items():
    res = ols(models[labels[0]])
    log_pred = res.predict(val)
    pred = np.expm1(log_pred).clip(lower=0)
    rows.append({"model": " = ".join(labels), "terms": len(key), "params": len(res.params),
                 "R2_adj": res.rsquared_adj, "AIC": res.aic, "BIC": res.bic,
                 "RMSE_val": rmse(pred, val["pickups"]), "MAE_val": meanabs(pred, val["pickups"]),
                 "RMSE_val_log": rmse(log_pred, np.log1p(val["pickups"]))})
    del res
rows.append({"model": "Baseline: same hour last week", "RMSE_val": rmse(val["lag168"], val["pickups"]),
             "MAE_val": meanabs(val["lag168"], val["pickups"]),
             "RMSE_val_log": rmse(val["log_lag168"], np.log1p(val["pickups"]))})
table = pd.DataFrame(rows).set_index("model")

BEST = table["RMSE_val"].drop("Baseline: same hour last week").idxmin()
print(f"lowest validation RMSE: {BEST}")
table

In [ ]:
steps = table.loc[table.index.str.match(r"S\d")]
labels = steps.index.str.split(": ").str[1]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True)
for ax, col, title in [(axes[0], "R2_adj", "Adjusted R² (train)"), (axes[1], "BIC", "BIC (train, lower is better)"),
                       (axes[2], "RMSE_val", "RMSE on validation weeks (pickups)")]:
    ax.plot(range(len(steps)), steps[col], marker="o", ms=5, color=BLUE)
    ax.set_title(title)
    ax.set_xticks(range(len(steps)), labels, rotation=60, ha="right", fontsize=8)
base_rmse = table.loc["Baseline: same hour last week", "RMSE_val"]
axes[2].axhline(base_rmse, color=INK_2, lw=1.2, ls="--")
axes[2].annotate("last-week baseline", (0, base_rmse), xytext=(4, 4), textcoords="offset points", color=INK_2, fontsize=9)
fig.suptitle("Forward stepwise path: each point adds one term", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

**Results.**
- **In-sample, every criterion prefers the full model.** Adjusted R² rises from 0.892 (S1) to 0.920 (S9), and AIC and
  BIC fall at every step.
- **Out-of-sample, the smallest models win.** S2 (only the two lags) has the lowest validation RMSE: **29.9 pickups**,
  vs 33.3 for the full model and 34.5 for the last-week baseline. S2 is also best on the log scale (0.370 vs 0.371),
  so the gap isn't caused by converting predictions back from logs.
- **Why:**
  - Last week's demand at the same zone and clock hour already contains the zone × weekday × hour pattern.
  - The dummies improve the training fit, but the improvement doesn't carry over to other weeks of the month. Month
    dummies cost the most (S2 → S3: +2.3 RMSE).
  - With 228k rows, BIC's penalty (12.3 per parameter) is far too small to stop terms that are statistically
    significant but don't help forecasts.

**Decision:**
- **S2 is the forecasting model** (lowest validation RMSE) and goes to Step 7.
- **The full model remains the explanatory model:** it is the one to quote for significance and effect sizes (for
  example the rain effect).

## Step 7: The chosen model on the test weeks

The model with the lowest validation RMSE is refitted on **train + validation** and scored once on the **test**
weeks, which no step above has used. Christmas and Thanksgiving fall in test weeks, so expect larger errors there.
Then the assumption plots are repeated for the chosen model's test residuals.

Note: `exp(ŷ) − 1` predicts a median-like value, so it slightly under-predicts the mean.

In [ ]:
chosen = models[BEST.split(" = ")[0]]
res_final = ols(chosen, data=pd.concat([train, val], ignore_index=True))
print(formula(chosen))
log_pred = res_final.predict(test)
test["model"] = np.expm1(log_pred).clip(lower=0)

summary = pd.DataFrame({
    "RMSE": [rmse(test["model"], test["pickups"]), rmse(test["lag168"], test["pickups"])],
    "MAE": [meanabs(test["model"], test["pickups"]), meanabs(test["lag168"], test["pickups"])],
}, index=[f"chosen model ({BEST})", "baseline: same hour last week"])
by_zone = test.groupby("zone").apply(lambda g: pd.Series({
    "mean pickups": g["pickups"].mean(), "RMSE model": rmse(g["model"], g["pickups"]),
    "RMSE baseline": rmse(g["lag168"], g["pickups"])}), include_groups=False)
by_zone["model wins"] = by_zone["RMSE model"] < by_zone["RMSE baseline"]
print(f"model beats the baseline in {by_zone['model wins'].mean():.0%} of the {len(by_zone)} zones")
summary

In [ ]:
by_zone.sort_values("mean pickups", ascending=False).head(15)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
diagnostics(axes, log_pred, np.log1p(test["pickups"]) - log_pred, "Chosen model, test weeks")
fig.tight_layout()
plt.show()

zone = by_zone["mean pickups"].idxmax()
week = test[(test["zone"] == zone) & (test["hour"].dt.month == 6)].set_index("hour")
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(week.index, week["pickups"], color=BLUE, label="actual", lw=2)
ax.plot(week.index, week["model"], color=ORANGE, label="chosen model", lw=1.6)
ax.plot(week.index, week["lag168"], color=AQUA, label="same hour last week", lw=1.6)
ax.set(title=f"Zone {zone}, June test week: hourly pickups", ylabel="pickups per hour")
ax.legend(frameon=False, ncols=3, loc="upper left")
fig.tight_layout()
plt.show()

**Results.**
- **Test weeks:** S2, refitted on train + validation, scores **RMSE 34.4 and MAE 19.3** pickups per zone-hour on the
  untouched test weeks. The last-week baseline scores RMSE 41.1 and MAE 22.4, so S2 is 16% lower on RMSE.
- **By zone:** S2 beats the baseline in **all 50 zones**.
- **Residuals:** test errors are larger than validation errors. The test weeks contain Memorial Day, Thanksgiving and
  Christmas, when last week's demand is a poor guide. The test residuals show the same pattern as in Step 1: variance
  falls as demand rises, and the left tail is heavy. Those tail points are hours far below forecast, such as holidays.
- **Zone 161 example:** in the June test week (Midtown Center), S2 follows the daily cycle closely. It smooths the
  extremes of last week's curve rather than copying them, but it can't anticipate unusual peaks (e.g. the
  2025-06-26 afternoon).

## Notes for the report

- **Two different questions, two models.** The full model (Step 4) explains demand: each term is significant after
  all the others. S2 (Step 6) forecasts demand best on new weeks. State which one a number comes from.
- **The lags assume a forecast made one day ahead.** For longer horizons, drop `log_lag24` and re-run Steps 5–7.
- **Day-clustered standard errors** are used in every significance test, because weather and the calendar are the
  same for all zones in a given hour.
- **Ideas for a better forecast:**
  - Add a holiday flag back for the test weeks.
  - Interact the lags with the hour of day.
  - Use more weeks of lagged demand.